## Part 2
This part focuses on using `spark` to analyze NFL data.

#### `pandas`-on-Spark
The code below imports the required modules.

In [1]:
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'

import pandas as pd
import numpy as np
import pyspark.pandas as ps

The code below reads in the weekly NFL data from the main filepath in JupyterHub.

In [2]:
ps.set_option("compute.fail_on_ansi_mode", False)
NFL = ps.read_csv("weekly_nfl_data.csv")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 15:13:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/20 15:13:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/20 15:13:46 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAP

Next, we use `.head()` to check out the first 5 rows of the `NFL` dataframe.

In [3]:
NFL.head()

26/03/20 15:13:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


The code below reports all the column names of the `NFL` dataframe.

In [4]:
NFL.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

Next, we only want to look at QB stats for the seasons 2005 to 2023 (inclusive). Additionally:
- We only want the following columns -
    - `player_display_name`
    - `season`
    - `week`
    - `completions`
    - `attempts`
    - `passing_yards`
    - `passing_tds`
    - `interceptions`
- For each combination of `player_display_name` and `season`, we will find the *sum* and *mean* of each of the statistical quantities in this subsetted dataset
- We will create two new variables by player/season combination:
    - `completion_percentage` = (sum of completions) / (sum of attempts)
    - `td_int_ratio` = (sum passing tds) / (sum interceptions)

The code below subsets `NFL` to look at regular season QB stats, specifically for seasons 2005 to 2023, inclusive. The code also subsets the columns of interest.

In [5]:
NFL_sub1 = NFL.loc[(NFL['position'] == 'QB') & 
                   (NFL['season_type'] == 'REG') & 
                   (NFL['season'].between(2005, 2023))] \
              [["player_display_name", "season", "week", "completions",
                "attempts", "passing_yards", "passing_tds", "interceptions"]]
NFL_sub1.head()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


Next, the code below calculates the sum and mean for each of the statistical quantities in this subsetted data for each combination of `player_display_name` and `season`.

In [6]:
NFL_sub2 = NFL_sub1.groupby(["player_display_name", "season"]) \
                    [["completions", "attempts", "passing_yards",
                      "passing_tds", "interceptions"]].agg(['mean', 'sum'])
NFL_sub2.head()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


completions        attempts      passing_yards         passing_tds     interceptions      
                                  mean  sum       mean  sum          mean     sum        mean sum          mean   sum
player_display_name season                                                                                           
Jake Delhomme       2006     20.230769  263  33.153846  431    215.769231  2805.0    1.307692  17      0.846154  11.0
Jake Plummer        2005     17.312500  277  28.500000  456    210.375000  3366.0    1.125000  18      0.437500   7.0
Matt Schaub         2006      3.600000   18   5.400000   27     41.600000   208.0    0.200000   1      0.400000   2.0
Vince Young         2006     12.266667  184  23.733333  356    146.600000  2199.0    0.800000  12      0.866667  13.0
Kerry Collins       2007      8.333333   50  13.666667   82     88.500000   531.0    0.000000   0      0.000000   0.0

Finally, we create the two new variables that were previously described! Then, they are added to our subsetted dataframe.

In [12]:
cp = NFL_sub2["completions", "sum"] / NFL_sub2["attempts", "sum"]
NFL_sub2["completion_percentage"] = cp

tdir = NFL_sub2["passing_tds", "sum"] / NFL_sub2["interceptions", "sum"]
NFL_sub2["td_int_ratio"] = tdir

NFL_sub2.dtypes

completions            mean    float64
                       sum       int64
attempts               mean    float64
                       sum       int64
passing_yards          mean    float64
                       sum     float64
passing_tds            mean    float64
                       sum       int64
interceptions          mean    float64
                       sum     float64
completion_percentage          float64
td_int_ratio                   float64
dtype: object

The final results of the above are saved in the `NFL_sub2` object.

Now, we take `NFL_sub2` and do the following:
- Subset the rows to only include player/season combinations wher ethe sum of attempts is at least 50.
- Sort the rows descending by completion_percentage and report the first 40 values!
- Sort the rows descending by td_int_ratio and report the first 40 values!

First, we subset the `NFL_sub2` object to only include the rows where the player/season combination sum of attempts is at least 50.

In [8]:
NFL_sub3 = NFL_sub2.loc[NFL_sub2["attempts", "sum"] >= 50]

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


Next, we sort the rows descending by `completion_percentage` and report the first 40 values.

In [9]:
NFL_sub3.sort_values("completion_percentage", ascending=False).head(40)

26/03/20 15:13:55 ERROR Executor: Exception in task 0.0 in stage 12.0 (TID 46)
org.apache.spark.SparkArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__div__" was called from
line 4 in cell [7]

	at org.apache.spark.sql.errors.QueryExecutionErrors$.divideByZeroError(QueryExecutionErrors.scala:203)
	at org.apache.spark.sql.errors.QueryExecutionErrors.divideByZeroError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.hashAgg_doAggregateWithKeysOutput_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEv

ArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__div__" was called from
line 4 in cell [7]


26/03/20 15:13:57 ERROR Executor: Exception in task 0.0 in stage 15.0 (TID 55)
org.apache.spark.SparkArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__div__" was called from
line 4 in cell [7]

	at org.apache.spark.sql.errors.QueryExecutionErrors$.divideByZeroError(QueryExecutionErrors.scala:203)
	at org.apache.spark.sql.errors.QueryExecutionErrors.divideByZeroError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.hashAgg_doAggregateWithKeysOutput_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEv

ArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__div__" was called from
line 4 in cell [7]


Finally, independent of the previous step, we sort the rows descending by `td_int_ratio` and report the first 40 values.

In [10]:
NFL_sub3.sort_values("td_int_ratio", ascending=False).head(40)

26/03/20 15:13:57 ERROR Executor: Exception in task 0.0 in stage 18.0 (TID 64)
org.apache.spark.SparkArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__div__" was called from
line 4 in cell [7]

	at org.apache.spark.sql.errors.QueryExecutionErrors$.divideByZeroError(QueryExecutionErrors.scala:203)
	at org.apache.spark.sql.errors.QueryExecutionErrors.divideByZeroError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.hashAgg_doAggregateWithKeysOutput_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEv

ArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__div__" was called from
line 4 in cell [7]


26/03/20 15:13:58 ERROR Executor: Exception in task 0.0 in stage 21.0 (TID 73)
org.apache.spark.SparkArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__div__" was called from
line 4 in cell [7]

	at org.apache.spark.sql.errors.QueryExecutionErrors$.divideByZeroError(QueryExecutionErrors.scala:203)
	at org.apache.spark.sql.errors.QueryExecutionErrors.divideByZeroError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.hashAgg_doAggregateWithKeysOutput_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEv

ArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__div__" was called from
line 4 in cell [7]


#### Spark SQL
The code below imports the required modules and creates a spark instance.

In [35]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

The code below reads in the weekly NFL data from the main filepath in JupyterHub.

In [36]:
NFL_spark = spark.read.load("weekly_nfl_data.csv",
                            format="csv",
                            sep=",",
                            inferSchema="true",
                            header="true")

Next, we use `.show(5)` to check out the first 5 rows of the `NFL_spark` dataframe.

In [37]:
NFL_spark.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

The code below reports all the column names of the `NFL_spark` dataframe.

In [18]:
NFL_spark.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

Next, we only want to look at QB stats for the seasons 2005 to 2023 (inclusive). Additionally:
- We only want the following columns -
    - `player_display_name`
    - `season`
    - `week`
    - `completions`
    - `attempts`
    - `passing_yards`
    - `passing_tds`
    - `interceptions`
- For each combination of `player_display_name` and `season`, we will find the *sum* and *mean* of each of the statistical quantities in this subsetted dataset
- We will create two new variables by player/season combination:
    - `completion_percentage` = (sum of completions) / (sum of attempts)
    - `td_int_ratio` = (sum passing tds) / (sum interceptions)

The code below subsets `NFL_spark` to meet all of the above criteria and also adds the two new variables. This is all accomplished in the one code chunk below. The results are saved as an object called `NFL_spark_ss`.

In [57]:
NFL_spark_ss = NFL_spark.filter((NFL_spark.position == 'QB') & (NFL_spark.season_type == 'REG') &
                                (NFL_spark.season.between(2005,2023))) \
                        .select(["player_display_name", "season", "week", "completions",
                                 "attempts", "passing_yards", "passing_tds", "interceptions"]) \
                        .groupby(["player_display_name", "season"]) \
                        .agg(sum("completions"), avg("completions"), sum("attempts"), avg("attempts"),
                             sum("passing_yards"), avg("passing_yards"), sum("passing_tds"), avg("passing_tds"),
                             sum("interceptions"), avg("interceptions")) \
                        .withColumn("completion_percentage", col("sum(completions)") / col("sum(attempts)")) \
                        .withColumn("td_int_ratio", col("sum(passing_tds)") / col("sum(interceptions)")) \
                        .show(5)

26/03/20 15:58:50 ERROR Executor: Exception in task 0.0 in stage 68.0 (TID 210)
org.apache.spark.SparkArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__truediv__" was called from
line 10 in cell [57]

	at org.apache.spark.sql.errors.QueryExecutionErrors$.divideByZeroError(QueryExecutionErrors.scala:203)
	at org.apache.spark.sql.errors.QueryExecutionErrors.divideByZeroError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.hashAgg_doAggregateWithKeysOutput_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCo

ArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__truediv__" was called from
line 10 in cell [57]


Now, we take `NFL_spark_ss` and do the following:
- Subset the rows to only include player/season combinations where the sum of attempts is at least 50.
- Sort the rows descending by completion_percentage and report the first 40 values!
- Sort the rows descending by td_int_ratio and report the first 40 values!

First, we show `NFL_spark_ss` subsetted to only include player/season combinations where the sum of attempts is at least 50. Then, we sort the rows descending by `completion_percentage` and report the first 40 values. This is all done in one code chunk.

In [59]:
NFL_spark_ss.filter(col("sum(attempts)") >= 50) \
            .sort(NFL_spark_ss.completion_percentage, ascending = False) \
            .show(40)

NameError: name 'NFL_spark_ss' is not defined

Finally, we show `NFL_spark_ss` subsetted to only include player/season combinations where the sum of attempts is at least 50. Then, we sort the rows descending by `td_int_ratio` and report the first 40 values. This is all done in one code chunk.

In [60]:
NFL_spark_ss.filter(col("sum(attempts)") >= 50) \
            .sort(NFL_spark_ss.td_int_ratio, ascending = False) \
            .show(40)

NameError: name 'NFL_spark_ss' is not defined